# Archon SigilAGI
## Global cyber-defense write-up

Inspired by the **Neural-Auto-Patch meme**, Archon SigilAGI is a policy-bounded guardian: it continuously observes public threat intelligence, verifies evidence, correlates enrolled assets, proposes bounded rescue actions, canary-tests them, and rolls back on failure. This notebook documents the complete real-world design and the executable control boundaries.

## What this system does

**Global Sentinel:** reads authoritative public feeds such as CISA KEV and NVD and publishes alerts for the world.

**Tenant Guardian:** acts only for organizations that explicitly enroll assets, SBOMs, telemetry, and response connectors.

**Hard boundary:** the model may extract facts and propose a typed plan. It cannot execute arbitrary shell, alter policy, patch firmware, or access credentials.

## One canonical live snapshot

All surfaces use the same versioned public snapshot: `https://raw.githubusercontent.com/BlockChain-BailBonds/archon-sigilagi/main/web/data/alerts.json`. GitHub Actions refreshes it every five minutes from CISA KEV; the dashboard and Kaggle demo read that same snapshot, so reports remain aligned and auditable.

In [ ]:
from dataclasses import dataclass
from enum import Enum
import json, hashlib, platform, subprocess, sys

print('Python:', sys.version.split()[0])
print('Platform:', platform.platform())
try:
    import torch
    print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
except Exception as exc:
    print('PyTorch unavailable:', exc)


## Trust boundaries

1. Public content is hostile data. Fetching is HTTPS-only, size-limited, hashed, and never executed.
2. Extraction is isolated from deployment authority.
3. Evidence is deterministic and source independence is explicit.
4. OPA-style policy is signed/versioned and fail-closed.
5. Executors are typed primitives with validation, expiry, canary, verification, rollback, and audit IDs.
6. Global alerts are public; tenant response requires owner authorization.

In [ ]:
AUTONOMOUS = {
    'block_ioc_temporarily', 'rate_limit_route', 'disable_feature_flag',
    'quarantine_workload', 'rollback_container_image'
}
IRREVERSIBLE = {
    'modify_bootloader', 'replace_trust_root', 'flash_firmware',
    'patch_kernel', 'destroy_encryption_key', 'delete_persistent_data'
}

def deterministic_score(vendor, kev, telemetry, applicability, reachability, reliability):
    return round(.30*vendor + .20*kev + .20*telemetry + .15*applicability + .10*reachability + .05*reliability, 4)

def policy(plan, score, independent_sources, telemetry_coverage, rollback_seconds, exact_version):
    reasons=[]
    if plan in IRREVERSIBLE or plan not in AUTONOMOUS: reasons.append('outside autonomous containment scope')
    if independent_sources < 2: reasons.append('needs two independent sources')
    if score < .85: reasons.append('score below canary threshold')
    if telemetry_coverage < .95: reasons.append('telemetry coverage below 95%')
    if rollback_seconds > 300: reasons.append('rollback window exceeds 300 seconds')
    if not exact_version: reasons.append('exact version applicability not proven')
    return {'allow': not reasons, 'mode': 'canary' if not reasons else 'escalate', 'reasons': reasons or ['all gates passed']}

print(policy('quarantine_workload', deterministic_score(.96,1,.99,1,1,.98), 2, .99, 120, True))
print(policy('flash_firmware', 1.0, 3, 1.0, 30, True))


## Data path

```text
CISA/NVD/vendor feeds → isolated ingest → normalized claim → evidence graph
→ SBOM and reachability correlation → deterministic policy → typed rescue plan
→ signed artifact/configuration → 1/5/20/50/100% canary → telemetry → rollback/audit
```

The global layer can protect the world’s awareness. The response layer can only protect enrolled assets because changing arbitrary internet systems would be unauthorized.

## Production launch gates

- 100% signed deployment artifacts
- 0 arbitrary command paths
- 0 model access to production credentials
- 100% reversible autonomous actions
- 95%+ telemetry coverage
- 99.9% rollback success in staging exercises
- immutable audit IDs for every decision

This is the difference between a superhero metaphor and a safe cyber-defense product.

## Public ecosystem watchlist

The companion demo reports public threat intelligence involving Hugging Face, X, GitHub, Cloudflare, Kubernetes, Docker, AWS, Microsoft, Google, OpenAI, Linux, Nginx, Apache, Python, npm, and PyPI. These are watchlist labels for advisories—not targets for probing. Rescue reports are produced only when an enrolled asset inventory and telemetry match the advisory.

## Kaggle execution note

This write-up is intentionally reproducible on Kaggle. The companion demo below exercises the live public-feed path when Internet is enabled and uses CUDA for batch risk calculations when a GPU is assigned. No credentials or offensive proof-of-concept code are used.